In [0]:
%pip install PyPDF2

#### Step 1: Discover pdf files

In [0]:
source_path = "/Volumes/agentic_catalog/agentic_schema/customer_service"
files = dbutils.fs.ls(source_path)
pdf_files = [f for f in files if f.name.endswith('.pdf')]

print(f"Below pdf files present: ")
for pdf in pdf_files:
    print(f" - {pdf.name}")

#### Step 2: Extract Text from each pdf
- Read each pdf file using PyPDF2
- Extract text from all pages
- Return product name and extracted text

In [0]:
import PyPDF2
import io

def extract_text_from_pdf(file_path):
    try:
        with open(file_path, 'rb') as f:
            pdf_bytes = f.read()

        pdf_reader = PyPDF2.PdfReader(io.BytesIO(pdf_bytes.encode('latin-1') if isinstance(pdf_bytes, str) else pdf_bytes))

        text_content = ""
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text_content += page.extract_text() + "\n"

        return text_content.strip()
    
    except Exception as e:
        print(f"Error extracting text from {file_path}: {str(e)}")
        return None
    
print("Text extraction function defined successfully!")
        

#### Step 3: Process all pdf files
Now we will iterate through each pdf file, extract the text, and prepare the data for our table

In [0]:
product_data = []

for pdf_file in pdf_files:
    print(f"Processing: {pdf_file.name}")

    ### Extract product name from file name & removing .pdf extention
    product_name = pdf_file.name.replace(".pdf", "") 

    ### Exract text from the pdf
    full_path = f"{source_path}/{pdf_file.name}"
    product_doc = extract_text_from_pdf(full_path)

    if product_doc:
        product_data.append({
            "product_name": product_name,
            "product_doc": product_doc
        })
        print(f"Successfully extracted {len(product_doc)} characters.")
    else:
        print("Failed to extract text!")

print(f"\n Total documents processed: {len(product_data)}")   
    

#### Step 4: Write Data into Delta Table
- Convert the extracted data into dataframe
- Write it in delta table `agentic_catalog.agentic_schema.product_docs`
- User `overwrite` mode to replace any existing data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("product_name", StringType(), False),
    StructField("product_doc", StringType(), True)
])

df = spark.createDataFrame(product_data, schema)

print(f"Dataframe created with {df.count()} rows.\n")
df.printSchema()

### Write into table
catalog_name = "agentic_catalog"
schema_name = "agentic_schema"
table_name = "product_docs"

df.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

print(f"Data written into table {catalog_name}.{schema_name}.{table_name}")

#### Step 5: Verify the table

In [0]:
%sql
select * from agentic_catalog.agentic_schema.product_docs limit 10